In [34]:
import mysql.connector


conn = mysql.connector.connect(
  host = "localhost",
  user = "root",
  password = "tarun@123",
  database="python_tut"

)





In [47]:
import csv

cursor = conn.cursor()

query = """
INSERT INTO movies (
    name, rating, genre, year, released, score, votes,
    director, writer, star, country, budget, gross,
    company, runtime
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

rows = []

with open("movies.csv", "r", encoding="latin-1") as file:

    reader = csv.reader(file, delimiter=";")
    next(reader)

    for row in reader:
        row = [value.strip() if value.strip() else None for value in row]
        rows.append(row)
try:
    cursor.executemany(query, rows)
    conn.commit()
    print(f"{cursor.rowcount} rows inserted successfully.")
except Exception as e:
    conn.rollback()
    print(e)
finally:
    print("Hello")


# cursor.close()
# conn.close()

4000 rows inserted successfully.
Hello


In [26]:
# Read CSV and insert
import csv

with open("movies.csv", "r", encoding="latin-1") as file:
    reader = csv.DictReader(file, delimiter=";")

    for i, row in enumerate(reader, start=2):  # Header is line 1
        if row["gross"] == " ":
            print(f"Row {i}: {row}")


In [72]:
import pandas as pd

mov = pd.read_csv(
    "movies.csv",
    sep=";",            # Your file uses semicolons
    encoding="latin-1"
)

# print(mov.head())


In [73]:
country_count = mov.groupby("country").size()

mov["avg_score_country"] = (
    mov.groupby("country")["score"]
       .transform("mean")
)
# print(country_count)
print(mov.head())

                                             name rating      genre  year  \
0                                     The Shining      R      Drama  1980   
1                                 The Blue Lagoon      R  Adventure  1980   
2  Star Wars: Episode V - The Empire Strikes Back     PG     Action  1980   
3                                       Airplane!     PG     Comedy  1980   
4                                      Caddyshack      R     Comedy  1980   

                        released  score      votes         director  \
0  June 13, 1980 (United States)    8.4   927000.0  Stanley Kubrick   
1   July 2, 1980 (United States)    5.8    65000.0   Randal Kleiser   
2  June 20, 1980 (United States)    8.7  1200000.0   Irvin Kershner   
3   July 2, 1980 (United States)    7.7   221000.0     Jim Abrahams   
4  July 25, 1980 (United States)    7.3   108000.0     Harold Ramis   

                    writer            star         country    budget  \
0             Stephen King  Jack Nicho

In [81]:
raw = pd.DataFrame({
 'Name': [' Alice ', 'BOB', 'charlie', None, 'Alice '],
 'Age': ['25', '30', 'unknown', '40', '25'],
 'Salary': [50000, 60000, None, 80000, 50000],
 'City': ['NYC', 'nyc', 'LA ', 'SF', 'NYC']
})
# Step 1: standardize text
raw['Name'] = raw['Name'].str.title()
raw['City'] = raw['City'].str.strip().str.upper()

# Step 2: coerce Age to numeric (bad strings -> NaN)
raw['Age'] = pd.to_numeric(raw['Age'], errors='coerce')
raw = raw.dropna(subset=['Name'])
raw = raw.drop_duplicates()
print(raw)


      Name   Age   Salary City
0   Alice   25.0  50000.0  NYC
1      Bob  30.0  60000.0  NYC
2  Charlie   NaN      NaN   LA
4   Alice   25.0  50000.0  NYC


In [87]:
import pandas as pd

# 1. Fixed: Added a 5th date to match the other columns' length (5 items)
sales = pd.DataFrame({
    'date': pd.to_datetime(['2024-01-05','2024-01-20','2024-02-10','2024-02-15','2024-02-20']),
 'region': ['East', 'West', 'East', 'West', 'East'],
 'product': ['A', 'B', 'A', 'A', 'B'],
 'amount': [1200, 800, 1500, 900, 1100]
})

# Total & average by region
print("--- Total & Average by Region ---")
print(sales.groupby('region')['amount'].agg(['sum', 'mean', 'count']))
print("\n")

# Monthly revenue trend
print("--- Monthly Revenue Trend ---")
print(sales.set_index('date')['amount'].resample('ME').sum())
print("\n")

# Best-selling product
print("--- Best-selling Product ---")
print(sales.groupby('product')['amount'].sum().nlargest(1))
print("\n")

# Region × product pivot 
# 2. Fixed: Completed 'margins=True'
print("--- Region × Product Pivot Table ---")
pivot = sales.pivot_table(index='region', columns='product', values='amount', aggfunc='sum', fill_value=0, margins=True)
print(pivot)

--- Total & Average by Region ---
         sum         mean  count
region                          
East    3800  1266.666667      3
West    1700   850.000000      2


--- Monthly Revenue Trend ---
date
2024-01-31    2000
2024-02-29    3500
Freq: ME, Name: amount, dtype: int64


--- Best-selling Product ---
product
A    3600
Name: amount, dtype: int64


--- Region × Product Pivot Table ---
product     A     B   All
region                   
East     2700  1100  3800
West      900   800  1700
All      3600  1900  5500


In [ ]:
emp = pd.DataFrame({
 'dept': ['Eng','Eng','Eng','Sales','Sales','HR'],
 'level': ['Jr','Sr','Sr','Jr','Sr','Sr'],
 'salary': [70,120,110,50,80,90]})
# Salary z-score WITHIN each department (transform keeps row count)
emp['salary_z'] = emp.groupby('dept')['salary'].transform(
 lambda s: (s - s.mean()) / s.std(ddof=0))
sales = pd.DataFrame({
 'date': pd.to_datetime(['2024-01-05','2024-01-20','2024-02-10','202
 'region': ['East','West','East','West','East'],
 'product':['A','B','A','A','B'],
 'amount': [1200, 800, 1500, 900, 1100]
})
# Total & average by region
sales.groupby('region')['amount'].agg(['sum','mean','count'])
# Monthly revenue trend
sales.set_index('date')['amount'].resample('M').sum()
# Best-selling product
sales.groupby('product')['amount'].sum().nlargest(1)
# Region × product pivot
sales.pivot_table(index='region', columns='product',
 values='amount', aggfunc='sum', fill_value=0, margins=T
# Departments whose average salary exceeds 80k
emp.groupby('dept').filter(lambda g: g['salary'].mean() > 80)
# Highest paid per department
emp.loc[emp.groupby('dept')['salary'].idxmax()]

SyntaxError: unterminated string literal (detected at line 9) (161087567.py, line 9)